# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Gradient Boosting

Sequentially correct errors for top accuracy on tabular data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

## Difference from Random Forests

## Step 1: Basic Gradient Boosting

In [ ]:
# Random Forest baseline
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_score = rf.score(X_test, y_test)

# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
gb_score = gb.score(X_test, y_test)

print(f"Random Forest:      {rf_score:.4f}")
print(f"Gradient Boosting:  {gb_score:.4f}")
print(f"Advantage:          {(gb_score - rf_score):+.4f}")

## Step 2: Learning Rate Trade-off

In [ ]:
# Learning rate controls step size
# Small learning rate: need more trees, more accurate
# Large learning rate: need fewer trees, risk overfitting

learning_rates = [0.001, 0.01, 0.05, 0.1, 0.2, 0.5]

print(f"{'Learning Rate':<15} {'Train':>8} {'Test':>8}")
print("-" * 32)

for lr in learning_rates:
    gb = GradientBoostingClassifier(
        n_estimators=100, learning_rate=lr, random_state=42
    )
    gb.fit(X_train, y_train)
    
    train_score = gb.score(X_train, y_train)
    test_score = gb.score(X_test, y_test)
    print(f"{lr:<15} {train_score:>8.4f} {test_score:>8.4f}")

## Step 3: n_estimators with Different Learning Rates

In [ ]:
# With small learning rate, need MORE trees
# With large learning rate, need FEWER trees

n_trees_list = [10, 25, 50, 100, 200, 500]
learning_rates = [0.01, 0.1, 0.5]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, lr in enumerate(learning_rates):
    test_scores = []
    
    for n_trees in n_trees_list:
        gb = GradientBoostingClassifier(
            n_estimators=n_trees, learning_rate=lr, random_state=42
        )
        gb.fit(X_train, y_train)
        test_scores.append(gb.score(X_test, y_test))
    
    axes[idx].semilogx(n_trees_list, test_scores, marker='o', linewidth=2)
    axes[idx].set_xlabel('Number of Trees')
    axes[idx].set_ylabel('Test Accuracy')
    axes[idx].set_title(f'Learning Rate = {lr}')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Smaller learning rates need more trees but often achieve better accuracy.")

## Step 4: Overfitting Risk

In [ ]:
# Gradient boosting can overfit with too many iterations
n_trees_list = range(10, 1001, 50)
train_scores = []
test_scores = []

for n_trees in n_trees_list:
    gb = GradientBoostingClassifier(
        n_estimators=n_trees, learning_rate=0.1, random_state=42
    )
    gb.fit(X_train, y_train)
    
    train_scores.append(gb.score(X_train, y_train))
    test_scores.append(gb.score(X_test, y_test))

plt.figure(figsize=(10, 6))
plt.plot(n_trees_list, train_scores, label='Training', marker='o')
plt.plot(n_trees_list, test_scores, label='Test', marker='s')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Gradient Boosting: Overfitting Risk')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Notice: Test accuracy peaks then starts declining (overfitting!)")
print("Random forests don't have this problem — more trees = always better.")

## Step 5: Feature Importance

In [ ]:
# Gradient boosting also provides feature importances
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)

importances = gb.feature_importances_
feature_names = X_train.columns

indices = np.argsort(importances)[::-1][:10]
print("Top 10 Most Important Features (Gradient Boosting):")
for rank, idx in enumerate(indices, 1):
    print(f"{rank:>2}. {feature_names[idx]:<30} {importances[idx]:.4f}")

## Step 6: Comparison with All Methods

In [ ]:
print("\n" + "="*60)
print("FINAL COMPARISON: All Ensemble Methods")
print("="*60)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

rf_acc = rf.score(X_test, y_test)
gb_acc = gb.score(X_test, y_test)

print(f"\nRandom Forest (100 trees):")
print(f"  Accuracy: {rf_acc:.4f}")
print(f"  Pros: Simple, no overfitting risk")
print(f"  Cons: May not achieve highest accuracy")

print(f"\nGradient Boosting (100 trees, lr=0.1):")
print(f"  Accuracy: {gb_acc:.4f}")
print(f"  Pros: Often highest accuracy")
print(f"  Cons: More tuning needed, can overfit")

print(f"\nAdvantage: {(gb_acc - rf_acc):+.4f}")